# 08 - Construcao da apresentacao (.pptx)

Monta `docs/entregas/Apresentacao_QuatroNorte.pptx` **inteiramente a partir** dos
artefatos gerados pelos notebooks (`reports/tables/` e `reports/figures/`). Nenhum
numero ou figura e digitado manualmente: o notebook e a fonte oficial e reproduzivel
de todo o conteudo. Requer `python-pptx`.

Sequencia de slides: contexto, empresa, problema, objetivos, hipoteses, referencial,
base consolidada, arquitetura, variavel resposta, correcao pela inflacao, dicionario,
estatisticas descritivas, histogramas, boxplots, relacao X-Y, ranking,
multicolinearidade, selecao de variaveis, modelagem, ML, comparacao, importancias,
discussao, limitacoes, trabalhos futuros e conclusao.


In [1]:
import pandas as pd
from pathlib import Path
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
TABLES = PROJECT_ROOT/"reports"/"tables"; FIG = PROJECT_ROOT/"reports"/"figures"
FIG_EDA = FIG/"eda"; OUT = PROJECT_ROOT/"docs"/"entregas"/"Apresentacao_QuatroNorte.pptx"
OUT.parent.mkdir(parents=True, exist_ok=True)

def rd(n):
    p = TABLES/n
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

stats = rd("03c_stats_ppt.csv").set_index("metrica")["valor"] if not rd("03c_stats_ppt.csv").empty else {}
inv = rd("00_inventario_base_consolidada.csv").set_index("metrica")["valor"] if not rd("00_inventario_base_consolidada.csv").empty else {}
metr = rd("05_metricas_modelos.csv"); best = rd("05_modelo_recomendado.csv")
imp = rd("05_importancia_permutacao_random_forest.csv"); hip = rd("06_hipoteses_final.csv")
sel = rd("05_selecao_variaveis.csv"); corr = rd("03b_correlacao_com_y.csv"); eta = rd("03b_eta_categoricas.csv")
dic = rd("02_dicionario_base_anual.csv"); vif = rd("03b_vif.csv"); rec = rd("06_recomendacoes_negocio.csv")
ev = rd("04_comparacao_nominal_deflacionado.csv")
def s(k, d="—"):
    try: return stats[k]
    except Exception: return d
cmpcfg = rd("05_comparacao_configuracoes.csv")


In [2]:
# ---- identidade visual ----
NAVY=RGBColor(0x14,0x2B,0x45); INK=RGBColor(0x1c,0x24,0x33); ORANGE=RGBColor(0xE8,0x71,0x3A)
BLUE=RGBColor(0x3b,0x6e,0xa5); PAPER=RGBColor(0xF6,0xF1,0xE7); GREY=RGBColor(0x5b,0x63,0x6e); WHITE=RGBColor(0xFF,0xFF,0xFF)
prs = Presentation(); prs.slide_width=Inches(13.333); prs.slide_height=Inches(7.5)
BLANK = prs.slide_layouts[6]; SW, SH = prs.slide_width, prs.slide_height

def slide(bg=PAPER):
    sl = prs.slides.add_slide(BLANK)
    r = sl.shapes.add_shape(1, 0, 0, SW, SH); r.fill.solid(); r.fill.fore_color.rgb=bg
    r.line.fill.background(); r.shadow.inherit=False
    sl.shapes._spTree.remove(r._element); sl.shapes._spTree.insert(2, r._element)
    return sl

def tb(sl, x, y, w, h, text, size=18, color=INK, bold=False, align=PP_ALIGN.LEFT, font="Calibri"):
    box = sl.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf=box.text_frame
    tf.word_wrap=True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i==0 else tf.add_paragraph(); p.alignment=align
        run=p.add_run(); run.text=line; f=run.font
        f.size=Pt(size); f.bold=bold; f.color.rgb=color; f.name=font
    return box

def header(sl, kicker, title):
    bar = sl.shapes.add_shape(1, Inches(0.6), Inches(0.55), Inches(0.16), Inches(0.9))
    bar.fill.solid(); bar.fill.fore_color.rgb=ORANGE; bar.line.fill.background(); bar.shadow.inherit=False
    tb(sl, 0.9, 0.5, 11.8, 0.4, kicker.upper(), 12, ORANGE, True)
    tb(sl, 0.9, 0.86, 11.8, 0.9, title, 27, NAVY, True)

def bullets(sl, x, y, w, h, items, size=16, color=INK, gap=6):
    box=sl.shapes.add_textbox(Inches(x),Inches(y),Inches(w),Inches(h)); tf=box.text_frame; tf.word_wrap=True
    for i,it in enumerate(items):
        p=tf.paragraphs[0] if i==0 else tf.add_paragraph(); p.space_after=Pt(gap)
        run=p.add_run(); run.text=("•  "+it); f=run.font; f.size=Pt(size); f.color.rgb=color; f.name="Calibri"
    return box

def pic_fit(sl, path, x, y, w, h):
    from PIL import Image as PImage
    p=Path(path)
    if not p.exists(): tb(sl,x,y,w,0.4,f"(figura ausente: {p.name})",11,GREY); return
    try:
        iw,ih=PImage.open(p).size; ar=iw/ih
    except Exception:
        ar=w/h
    bw,bh=w,w/ar
    if bh>h: bh=h; bw=h*ar
    sl.shapes.add_picture(str(p), Inches(x+(w-bw)/2), Inches(y+(h-bh)/2), Inches(bw), Inches(bh))

def pic_grid(sl, paths, x, y, w, h, cols=2, rows=2, gap=0.2):
    cw=(w-gap*(cols-1))/cols; ch=(h-gap*(rows-1))/rows
    for i,p in enumerate(paths[:cols*rows]):
        r,c=divmod(i,cols); pic_fit(sl, p, x+c*(cw+gap), y+r*(ch+gap), cw, ch)

def table(sl, df, x, y, w, h, fs=11, maxrows=12, header_bg=NAVY):
    df=df.head(maxrows); nr,nc=df.shape
    gt=sl.shapes.add_table(nr+1, nc, Inches(x),Inches(y),Inches(w),Inches(h)).table
    for j,col in enumerate(df.columns):
        cell=gt.cell(0,j); cell.text=str(col); cell.fill.solid(); cell.fill.fore_color.rgb=header_bg
        pr=cell.text_frame.paragraphs[0]; pr.runs[0].font.size=Pt(fs); pr.runs[0].font.bold=True; pr.runs[0].font.color.rgb=WHITE
    for i in range(nr):
        for j in range(nc):
            cell=gt.cell(i+1,j); cell.text=str(df.iloc[i,j])
            cell.fill.solid(); cell.fill.fore_color.rgb=WHITE if i%2==0 else RGBColor(0xEE,0xEA,0xDE)
            pr=cell.text_frame.paragraphs[0]
            if pr.runs: pr.runs[0].font.size=Pt(fs); pr.runs[0].font.color.rgb=INK
    return gt

def divider(kicker, title):
    sl=slide(NAVY)
    bar=sl.shapes.add_shape(1, Inches(0.9), Inches(3.0), Inches(0.9), Inches(0.14))
    bar.fill.solid(); bar.fill.fore_color.rgb=ORANGE; bar.line.fill.background(); bar.shadow.inherit=False
    tb(sl,0.9,2.3,11,0.5,kicker.upper(),14,ORANGE,True)
    tb(sl,0.9,3.3,11.6,1.6,title,34,WHITE,True)
    return sl
print("helpers ok")


helpers ok


In [3]:
# ============ SLIDES ============
best_pred = best[best['cenario']=='preditivo'].iloc[0] if not best.empty else None
best_expl = best[best['cenario']=='explicativo'].iloc[0] if not best.empty else None

# 1. Capa
sl=slide(NAVY)
tb(sl,0.9,2.1,11.6,0.5,"MBA · QUATRO NORTE CONSULTING · GRUPO 01",13,ORANGE,True)
tb(sl,0.9,2.7,11.6,1.6,"Custo anual de manutencao por carreta",40,WHITE,True)
tb(sl,0.9,4.2,11.6,0.9,"Fatores determinantes e modelagem do custo anual (CAD/ano, real) da frota\npropria de leasing/rental no Canada — janela 2020–2025",17,PAPER)
tb(sl,0.9,6.4,11.6,0.5,"Marlon Wenzel · Jeison Lima · Rodrigo Queiroz · Giovani Cani",13,PAPER)

# 2. Contexto / Empresa
sl=slide(); header(sl,"01 · Contexto","A empresa e a operacao")
bullets(sl,0.9,1.9,7.2,4.6,[
 "Empresa de leasing/rental de carretas no Canada (secas e refrigeradas, ate 53').",
 "Manutencao propria: oficinas registram ordens de servico com mao de obra e pecas.",
 "Todos os custos em dolares canadenses (CAD); operacao integralmente no Canada.",
 "Objetivo de negocio: apoiar orcamento anual e priorizacao de manutencao da frota.",
 "Fonte unica do estudo: base consolidada de OS 'fato_wo_ml' (2020–2025).",
],16)
for i,(k,v) in enumerate([("Carretas",str(int(float(inv.get('carretas_distintas',s('carretas',0)))))),
                          ("Ordens de servico",f"{int(float(inv.get('linhas_os',0))):,}".replace(',','.')),
                          ("Custo interno (real)",f"CAD {s('custo_total_real_mi')} mi"),
                          ("Periodo","2020–2025")]):
    bx=8.5; by=1.9+i*1.15
    c=sl.shapes.add_shape(1,Inches(bx),Inches(by),Inches(4.0),Inches(1.0)); c.fill.solid(); c.fill.fore_color.rgb=WHITE
    c.line.color.rgb=NAVY; c.shadow.inherit=False
    tb(sl,bx+0.2,by+0.08,3.7,0.4,v,22,ORANGE,True); tb(sl,bx+0.2,by+0.58,3.7,0.35,k,12,GREY)

# 3. Problema
sl=slide(); header(sl,"02 · Problema","A pergunta de pesquisa")
c=sl.shapes.add_shape(1,Inches(0.9),Inches(1.9),Inches(11.5),Inches(1.3)); c.fill.solid(); c.fill.fore_color.rgb=NAVY; c.line.fill.background(); c.shadow.inherit=False
tb(sl,1.2,2.05,11,1.0,"Quais fatores mais influenciam o CUSTO ANUAL de manutencao por carreta —\ne como estima-lo a partir das caracteristicas operacionais, historicas e estruturais da frota?",18,WHITE,True)
bullets(sl,0.9,3.5,11.5,3.2,[
 "Variavel de interesse: custo anual de manutencao por carreta (CAD/ano), em valores reais.",
 "Grao de analise: carreta × ano.",
 "Escopo: todo o custo interno absorvido pela empresa (preventiva + corretiva).",
 "Correcao monetaria obrigatoria pela inflacao canadense (CPI) para comparar anos em termos reais.",
],16)

# 4. Objetivos
sl=slide(); header(sl,"03 · Objetivos","Objetivo geral e especificos")
tb(sl,0.9,1.85,11.6,1.1,"Geral: analisar os fatores que influenciam o custo anual de manutencao por carreta (CAD, corrigido pela inflacao), identificando as variaveis de maior capacidade explicativa e desenvolvendo modelos estatisticos e de ML para estima-lo.",15,INK,True)
bullets(sl,0.9,3.1,11.6,3.4,[
 "Consolidar a analise a partir da base unica de OS (etapa de preparacao ja realizada).",
 "Definir a variavel resposta anual e corrigi-la pelo CPI do Canada.",
 "Realizar EDA rigorosa (univariada, relacao com Y, ranking) das variaveis candidatas.",
 "Selecionar variaveis de forma fundamentada (associacao, multicolinearidade, dominio).",
 "Desenvolver e avaliar modelos estatisticos e de Machine Learning (split temporal).",
],16)

# 5. Hipoteses
sl=slide(); header(sl,"04 · Hipoteses","Hipoteses de trabalho")
if not hip.empty:
    table(sl, hip[["hipotese","enunciado","veredito"]], 0.9,1.9,11.6,4.6, fs=11, maxrows=10)
tb(sl,0.9,6.7,11.6,0.5,"H6 (contrato) tornou-se testavel em 2026-08-16, quando os dados de contrato passaram a integrar a fonte unica; desdobrada em H6a (duracao) e H6b (tipo).",11,GREY)


In [4]:
# 6. Referencial teorico (preservado)
sl=slide(); header(sl,"05 · Referencial teorico","O que a literatura mostra")
bullets(sl,0.9,1.9,11.6,4.8,[
 "Katreddi et al. (2023) — ML para custo de manutencao por milha em caminhoes (Super Learner, R²=97%); variaveis operacionais e ensembles capturam relacoes nao lineares.",
 "Katreddi et al. (2023) — Mixed Effects Random Forest: captura diferencas sistematicas entre grupos de ativos; motiva incluir categoricas (subtipo, montadora, reefer).",
 "Sun et al. (2024) — previsao por dados historicos de manutencao; reforca que o historico e insumo suficiente para prever custos futuros.",
 "Adekitan et al. (2018) — ANN com R=0,766; sinal preditivo de variaveis de uso mesmo com dados limitados.",
],15)
tb(sl,0.9,6.7,11.6,0.4,"As referencias favorecem modelos de arvore/ensemble, coerente com os resultados deste estudo.",11,GREY)

# 7. Base consolidada + arquitetura
sl=slide(); header(sl,"06 · Base de dados","Base consolidada unica (Single Source of Truth)")
bullets(sl,0.9,1.9,7.0,4.6,[
 "Fonte unica: fato_wo_ml_2020-01-01_to_2025-12-31.csv (1 linha = 1 OS).",
 "Extracao SQL, modelo estrela, joins e feature engineering: ETAPA ANTERIOR de preparacao.",
 "O estudo NAO reconstroi a base, nao refaz joins nem o feature engineering.",
 f"{int(float(inv.get('linhas_os',0))):,}".replace(',','.')+" OS · "+str(int(float(inv.get('carretas_distintas',0))))+" carretas · 2020–2025.",
 "VMRS: codigo padronizado do sistema reparado (PM, freios, pneus, reefer...).",
],15)
tb(sl,8.2,1.9,4.4,4.8,"Arquitetura metodologica\n\nBase consolidada (CSV)\n↓ Validacao de qualidade\n↓ Variavel resposta anual\n↓ Correcao CPI Canada\n↓ EDA (uni/bivariada)\n↓ Correlacao e testes\n↓ Multicolinearidade/VIF\n↓ Ranking\n↓ Selecao de variaveis\n↓ Modelagem (estat.+ML)\n↓ Avaliacao / discussao",13,NAVY,True)

# 8. Variavel resposta
sl=slide(); header(sl,"07 · Variavel resposta","Y = custo anual de manutencao por carreta")
c=sl.shapes.add_shape(1,Inches(0.9),Inches(1.9),Inches(11.5),Inches(1.15)); c.fill.solid(); c.fill.fore_color.rgb=NAVY; c.line.fill.background(); c.shadow.inherit=False
tb(sl,1.2,2.05,11,0.9,"Y = soma do custo interno da carreta no ano (CAD), deflacionado pelo CPI Canada (dez/2025)\nGrao: carreta × ano",17,WHITE,True)
bullets(sl,0.9,3.3,11.6,3.2,[
 f"Media CAD {s('y_media')}/ano · mediana CAD {s('y_mediana')}/ano · p90 CAD {s('y_p90')}.",
 f"Assimetria {s('assimetria')} (cauda longa), tratada por log1p e modelos robustos.",
 f"Apenas {s('share_y_zero_pct')}% de carreta-anos com custo zero: o grao anual praticamente elimina a zero-inflacao.",
 f"Custo interno total: CAD {s('custo_total_nominal_mi')} mi nominal / CAD {s('custo_total_real_mi')} mi real (dez/2025).",
],16)

# 9. Correcao pela inflacao
sl=slide(); header(sl,"08 · Correcao pela inflacao","Custos reais pelo CPI do Canada")
bullets(sl,0.9,1.9,5.6,4.4,[
 "Todos os custos em CAD, corrigidos pelo CPI all-items do Canada (StatCan, vetor v41690973).",
 "Base de comparacao: dezembro de 2025.",
 "Inflacao acumulada no periodo: ~20% (2020→2025).",
 "A correcao isola mudancas REAIS de custo da mera perda do poder de compra.",
 "O deflator e canadense (CPI), coerente com custos registrados em CAD.",
],15)
pic_fit(sl, FIG/"04_nominal_vs_deflacionado.png", 6.7, 1.9, 6.0, 4.6)


In [5]:
# 10. Dicionario das variaveis
sl=slide(); header(sl,"Variaveis","Dicionario das variaveis candidatas")
tb(sl,0.9,1.55,11.6,0.4,"Universo derivavel da fonte unica (29 colunas), ja incluindo contrato. Mao de obra, pecas e tipo_contrato (RENTAL/LEASE) exigiriam outras tabelas.",11,GREY)
if not dic.empty:
    table(sl, dic[["variavel","tipo/papel"]].rename(columns={"tipo/papel":"tipo / papel"}), 0.7,2.0,5.9,4.9, fs=8, maxrows=18)
    table(sl, dic[["variavel","tipo/papel"]].iloc[18:].rename(columns={"tipo/papel":"tipo / papel"}), 6.8,2.0,5.9,4.9, fs=8, maxrows=18)

# 11. Estatisticas descritivas
sl=slide(); header(sl,"EDA","Estatisticas descritivas")
est=rd("03b_estatisticas_descritivas.csv")
if not est.empty:
    cols=[c for c in ["variavel","N","media","mediana","desvio_padrao","min","max","assimetria"] if c in est.columns]
    table(sl, est[cols], 0.7,1.9,12.0,4.8, fs=9, maxrows=13)

# 12. Histogramas (montagem)
sl=slide(); header(sl,"EDA","Histogramas e distribuicoes")
pic_grid(sl,[FIG_EDA/"quant_custo_ano_real.png", FIG_EDA/"quant_km_rodado_ano.png",
             FIG_EDA/"quant_custo_ano_anterior.png", FIG_EDA/"quant_n_os_ano_anterior.png"],
         0.7,1.7,12.0,5.2,2,2)

# 13. Boxplots por categoria
sl=slide(); header(sl,"EDA","Boxplots do custo anual por categoria")
pic_grid(sl,[FIG_EDA/"quali_flag_refrigerado.png", FIG_EDA/"quali_unit_subtype.png",
             FIG_EDA/"quali_cod_montadora.png", FIG_EDA/"quali_vmrs_predominante_ano.png"],
         0.7,1.7,12.0,5.2,2,2)

# 14. Relacao individual X-Y
sl=slide(); header(sl,"EDA","Relacao individual de cada variavel com Y")
if not corr.empty:
    table(sl, corr[["variavel","pearson","spearman","papel"]], 0.7,1.9,6.0,4.8, fs=9, maxrows=13)
if not eta.empty:
    table(sl, eta[["variavel","eta","n_categorias"]], 7.0,1.9,5.7,4.8, fs=9, maxrows=11)
tb(sl,0.7,6.75,12,0.3,"Quantitativas: Pearson/Spearman. Categoricas: eta (ANOVA).",10,GREY)

# 15. Ranking
sl=slide(); header(sl,"EDA","Ranking de associacao com o custo anual")
pic_fit(sl, FIG_EDA/"ranking_associacao_y.png", 0.9,1.7,7.4,5.2)
bullets(sl,8.4,2.0,4.2,4.6,[
 "Uso (km) e historico defasado lideram entre as quantitativas.",
 "Refrigerado e subtipo lideram entre as categoricas.",
 "Idade tem efeito direto fraco.",
 "Componentes aritmeticos de Y sao excluidos do ranking de explicadores.",
],14)

# 16. Multicolinearidade
sl=slide(); header(sl,"EDA","Multicolinearidade — matriz e VIF")
pic_fit(sl, FIG_EDA/"matriz_spearman.png", 0.7,1.7,7.2,5.2)
if not vif.empty:
    table(sl, vif, 8.2,1.9,4.4,4.8, fs=10, maxrows=12)

# 17. Selecao de variaveis
sl=slide(); header(sl,"Selecao","Selecao das variaveis do modelo")
if not sel.empty:
    table(sl, sel[["variavel","decisao"]], 0.7,1.9,6.0,4.9, fs=8, maxrows=17)
    table(sl, sel[["variavel","decisao"]].iloc[17:], 6.9,1.9,6.0,4.9, fs=8, maxrows=17)
tb(sl,0.7,6.85,12,0.3,"Criterios: ranking, VIF, redundancia, vazamento temporal e coerencia de dominio.",10,GREY)


In [6]:
# 18. Modelagem — tecnicas
sl=slide(); header(sl,"09 · Modelagem","Tecnicas e desenho experimental")
bullets(sl,0.9,1.9,11.6,4.6,[
 "Estatistica: regressao linear multipla, ridge, regressao polinomial.",
 "Machine Learning: arvore de decisao, Random Forest, Gradient Boosting, KNN.",
 "Alvo transformado por log1p (assimetria); metricas na escala original (CAD/ano).",
 "Split temporal: treino 2020–2024, teste 2025 (evita vazamento entre periodos).",
 "Dois cenarios: explicativo (inclui uso do ano) e preditivo (so historico defasado + atributos).",
 "Populacao: contrato com manutencao inclusa (MAINT), aplicada como flag — o cenario sem filtro e mantido como baseline.",
 "Tres configuracoes comparadas para separar o efeito do filtro MAINT do efeito das variaveis de contrato.",
 "Metricas: R², RMSE e MAE.",
],16)

# 19. Comparacao dos modelos
sl=slide(); header(sl,"09 · Modelagem","Comparacao dos modelos (teste 2025)")
if not metr.empty:
    table(sl, metr, 0.7,1.8,7.4,5.0, fs=10, maxrows=14)
if best_pred is not None:
    _cp = cmpcfg[cmpcfg["cenario"] == "preditivo"].iloc[0] if not cmpcfg.empty else None
    _itens = [
     f"Preditivo recomendado: {best_pred['modelo']}",
     f"R² = {best_pred['r2']} · RMSE = {best_pred['rmse']}",
     f"MAE = {best_pred['mae']} CAD/ano",
     f"Explicativo (melhor): R² = {best_expl['r2']}",
     "Arvores/ensembles superam modelos lineares.",
    ]
    if _cp is not None:
        _itens += [
         f"Efeito do filtro MAINT: {float(_cp['delta_r2_filtro_maint']):+.4f} de R².",
         f"Efeito das variaveis de contrato: {float(_cp['delta_r2_contrato']):+.4f} de R² — praticamente nulo.",
        ]
    bullets(sl,8.4,2.0,4.2,4.6,_itens,13)

# 20. Importancia das variaveis
sl=slide(); header(sl,"09 · Modelagem","Variaveis mais importantes (permutacao)")
if not imp.empty:
    top=imp.head(10).iloc[::-1]
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(7.6,5.0)); ax.barh(top['variavel'], top['importancia'], color="#3b6ea5")
    ax.set_xlabel("Importancia por permutacao (queda de R²)"); ax.set_title("Fatores do custo anual (cenario preditivo)")
    fig.tight_layout(); tmp=FIG/"05_importancia_permutacao.png"; fig.savefig(tmp,dpi=140); plt.close(fig)
    pic_fit(sl, tmp, 0.7,1.7,7.6,5.2)
    table(sl, imp.head(8)[["variavel","importancia"]], 8.5,1.9,4.1,4.6, fs=10, maxrows=8)

# 21. Discussao
sl=slide(); header(sl,"Discussao","Discussao dos resultados")
bullets(sl,0.9,1.9,11.6,4.8,[
 "O custo anual real por carreta cresceu de forma consistente (2020→2025) mesmo apos deflacao — aumento real, nao inflacionario.",
 "Refrigeracao e o fator estrutural mais forte; subtipo e montadora tambem separam custos.",
 "O historico de manutencao (OS e custo de anos anteriores) e o principal preditor sem vazamento.",
 "O uso acumulado/quilometragem esta entre os maiores associados ao custo.",
 "Idade isolada tem efeito fraco; opera principalmente via historico e uso.",
 "Contrato (H6): efeito fraco. A duracao de contrato quase nao acrescenta poder preditivo, e o tipo contratual separa pouco os custos — hipotese testada, nao assumida.",
 f"O modelo preditivo (R² ≈ {best_pred['r2'] if best_pred is not None else '—'}) e util para orcamento e priorizacao de frota.",
],15)

# 22. Limitacoes
sl=slide(); header(sl,"Limitacoes","Limitacoes metodologicas")
bullets(sl,0.9,1.9,11.6,4.8,[
 "Fonte unica: contrato ja incorporado; seguem ausentes mao de obra, pecas e tipo_contrato (RENTAL/LEASE).",
 "Populacao restrita a MAINT: o ganho de R² do filtro reflete amostra mais homogenea, nao melhora de previsao.",
 "franquia_km_mensal_contrato descartada (99,8% de zeros); NET/MIX somam 2,7% das OS, o que limita conclusoes sobre esses regimes.",
 "cod_cliente nao modelado (597 categorias): risco de o modelo memorizar o cliente em vez de explicar o custo.",
 "km derivado do odometro nas OS; resets/ruido tratados por regra, com aproximacao.",
 "Provincia parcial (~54%); regiao usada como proxy geografica.",
 "Estornos (custos negativos) excluidos; span ativo assume presenca entre 1ª e ultima OS.",
],15)

# 23. Trabalhos futuros + conclusao
sl=slide(); header(sl,"Futuro","Trabalhos futuros")
bullets(sl,0.9,1.9,11.6,2.6,[
 "Integrar mao de obra, pecas e tipo_contrato (RENTAL/LEASE) — contrato de manutencao ja foi incorporado.",
 "Testar modelos de efeitos mistos por grupos de ativos (montadora/subtipo).",
 "Incorporar quilometragem planejada e telemetria (GPS) como exposicao.",
],15)
c=sl.shapes.add_shape(1,Inches(0.9),Inches(4.7),Inches(11.5),Inches(2.0)); c.fill.solid(); c.fill.fore_color.rgb=NAVY; c.line.fill.background(); c.shadow.inherit=False
tb(sl,1.2,4.9,11,1.7,"Conclusao\nSobre a base unica reextraida (29 colunas), em CAD corrigidos pelo CPI do Canada e no grao carreta x ano, o custo anual real e explicado sobretudo por refrigeracao, historico de manutencao e uso. As variaveis de contrato, incorporadas nesta fase, foram testadas e mostraram efeito fraco — resultado que decorre de teste, nao de suposicao.",14,WHITE,True)
prs.save(OUT)
n=len(prs.slides._sldIdLst)
print(f"OK: {OUT} — {n} slides")


OK: C:\Users\rodri\OneDrive\Área de Trabalho\quatro_norte-master\docs\entregas\Apresentacao_QuatroNorte.pptx — 23 slides
